# Comparación XGBoost–InterGATE: recuperación biológica Basal-like

Este análisis reutiliza los cinco XGBoost congelados del benchmark pareado y
los compara con el núcleo génico Basal-like estricto 5/5 de InterGATE. No se
reentrena ningún modelo.

El protocolo queda fijado antes de calcular TreeSHAP:

1. mismo universo ordenado de 7.131 genes;
2. contraste TreeSHAP de desarrollo para el margen Basal-like:
   media en Basal-like menos media en el resto;
3. agregación por rango con el mismo peso para cada cohorte y después para
   cada una de las cinco semillas;
4. panel XGBoost principal de 288 genes, igual al núcleo InterGATE;
5. firmas direccionales, PAM50 excluido, Hallmark/C2 y nulo génico común;
6. las cohortes externas se analizan únicamente después de escribir el
   bloqueo de desarrollo.

La unidad comparable son genes y programas. Las interacciones TreeSHAP son
interacciones del clasificador y no se reinterpretan como aristas moleculares.



In [ ]:
from pathlib import Path
import gc
import hashlib
import json
import os
import shutil
import sys

import pandas as pd
from IPython.display import display

here = Path.cwd().resolve()
candidates = [here, *here.parents]
PROJECT_ROOT = next(
    (path for path in candidates if (path / "intergate").is_dir()),
    None,
)
if PROJECT_ROOT is None:
    PROJECT_ROOT = next(
        (
            path / "code_snapshot"
            for path in candidates
            if (path / "code_snapshot" / "intergate").is_dir()
        ),
        None,
    )
if PROJECT_ROOT is None:
    raise FileNotFoundError(
        "No se encuentra intergate/. Abra este notebook dentro del proyecto."
    )
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print("Raíz del proyecto:", PROJECT_ROOT)



## 1. Rutas y parámetros bloqueados

Normalmente no es necesario modificar nada si el notebook 08a se ejecutó en
este mismo proyecto. Las variables de entorno permiten señalar otra ubicación
sin editar el código.



In [ ]:
DATA_PREPARE_DIR = Path(
    os.environ.get("INTERGATE_DATA_PREPARE_DIR", PROJECT_ROOT / "data_prepare")
).expanduser().resolve()
ARTIFACTS_ROOT = Path(
    os.environ.get(
        "INTERGATE_ARTIFACTS_ROOT", PROJECT_ROOT / "artifacts" / "intrinsic4"
    )
).expanduser().resolve()
CACHE_ROOT = Path(
    os.environ.get("INTERGATE_CACHE_ROOT", PROJECT_ROOT / "cache")
).expanduser().resolve()
EXTERNAL_DATA_DIR = Path(
    os.environ.get(
        "INTERGATE_EXTERNAL_DATA_DIR", PROJECT_ROOT / "data" / "external"
    )
).expanduser().resolve()
RESULTS_ROOT = Path(
    os.environ.get("INTERGATE_RESULTS_ROOT", PROJECT_ROOT / "results_v8")
).expanduser().resolve()

BENCHMARK_ROOT = RESULTS_ROOT / "08a_paired_predictive_benchmark"
BIOLOGY_RESULTS_DIR = RESULTS_ROOT / "05_biology_validation"
RESOURCE_DIR = PROJECT_ROOT / "resources_biological"
OUTPUT_DIR = RESULTS_ROOT / "08c_xgboost_intergate_biology"
PROTOCOL_SOURCE = (
    PROJECT_ROOT / "protocols" / "xgboost_intergate_biology_locked_v1.json"
)
INTERGATE_CORE_FILE = (
    PROJECT_ROOT / "protocols" / "intergate_basal_strict_5of5_288_genes.csv"
)

N_MATCHED_DRAWS = 1000
RANDOM_SEED = 20260828
TREE_SHAP_BATCH_SIZE = 256
TREE_SHAP_DEVICE = "cpu"  # No compite con las permutaciones que usan la GPU.
RUN_EXTERNAL = True

# Deben fijarse antes de importar intergate.config.CFG.
os.environ["INTERGATE_DATA_PREPARE_DIR"] = str(DATA_PREPARE_DIR)
os.environ["INTERGATE_ARTIFACTS_ROOT"] = str(ARTIFACTS_ROOT)
os.environ["INTERGATE_CACHE_ROOT"] = str(CACHE_ROOT)
os.environ["INTERGATE_EXTERNAL_DATA_DIR"] = str(EXTERNAL_DATA_DIR)

display(pd.Series({
    "benchmark": BENCHMARK_ROOT,
    "biology_results": BIOLOGY_RESULTS_DIR,
    "resources": RESOURCE_DIR,
    "output": OUTPUT_DIR,
    "matched_null_draws": N_MATCHED_DRAWS,
    "TreeSHAP_batch_size": TREE_SHAP_BATCH_SIZE,
    "TreeSHAP_device": TREE_SHAP_DEVICE,
    "run_external": RUN_EXTERNAL,
}, name="value").to_frame())



## 2. Preflight y copia inmutable del protocolo



In [ ]:
import numpy as np
import scipy
import sklearn
import xgboost

from intergate.config import CFG
from intergate.runtime_setup import (
    assert_transport_protocol,
    assert_v8_protocol,
    prepare_runtime,
)
from intergate.xgboost_intergate_biology import (
    DEFAULT_SEEDS,
    PANEL_SIZE,
    PROTOCOL_ID,
    finalize_comparison,
    load_intergate_basal_core,
    preflight_frozen_xgboost,
    run_development_comparison,
    run_external_validation,
)

assert_v8_protocol(CFG)
assert_transport_protocol(CFG)

required = [
    PROTOCOL_SOURCE,
    INTERGATE_CORE_FILE,
    BENCHMARK_ROOT / "development_benchmark_manifest.json",
    BIOLOGY_RESULTS_DIR / "locked_signature_manifest.json",
    RESOURCE_DIR / "PAM50_symbols.txt",
]
for seed in DEFAULT_SEEDS:
    required.append(
        BENCHMARK_ROOT / "frozen_tabular_models" / f"seed_{seed}" / "xgboost.joblib"
    )
missing = [path for path in required if not path.is_file()]
if missing:
    raise FileNotFoundError(
        "Faltan entradas obligatorias:\n- " + "\n- ".join(map(str, missing))
    )

protocol = json.loads(PROTOCOL_SOURCE.read_text(encoding="utf-8"))
if protocol.get("protocol_id") != PROTOCOL_ID:
    raise RuntimeError("El protocolo JSON no coincide con el módulo de análisis.")
if int(protocol["development_ranking"]["size_matched_panel"]) != PANEL_SIZE:
    raise RuntimeError("El tamaño del panel no coincide con el protocolo.")
core_digest = hashlib.sha256(INTERGATE_CORE_FILE.read_bytes()).hexdigest()
if core_digest != protocol["intergate_panel"]["locked_file_sha256"]:
    raise RuntimeError("El panel InterGATE bloqueado ha cambiado.")

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
protocol_copy = OUTPUT_DIR / "PRESPECIFIED_PROTOCOL.json"
if protocol_copy.exists():
    if protocol_copy.read_bytes() != PROTOCOL_SOURCE.read_bytes():
        raise RuntimeError("El protocolo preexistente es distinto; use otra salida.")
else:
    shutil.copy2(PROTOCOL_SOURCE, protocol_copy)

core, core_source = load_intergate_basal_core(
    RESULTS_ROOT, locked_core_file=INTERGATE_CORE_FILE
)
print(f"Núcleo InterGATE localizado: {len(core)} genes en {core_source}")
print("Protocolo bloqueado:", protocol_copy)
display(pd.Series({
    "python": sys.version.split()[0],
    "numpy": np.__version__,
    "pandas": pd.__version__,
    "scipy": scipy.__version__,
    "scikit-learn": sklearn.__version__,
    "xgboost": xgboost.__version__,
}, name="version").to_frame())



## 3. Desarrollo: TreeSHAP, panel de 288 genes y recuperación de programas

Esta fase no carga las matrices externas. Primero calcula y escribe el panel
XGBoost; solo después crea `DEVELOPMENT_LOCK.json`.



In [ ]:
development_lock = OUTPUT_DIR / "DEVELOPMENT_LOCK.json"
if development_lock.is_file():
    print("La fase de desarrollo ya está bloqueada; se reutilizan sus resultados.")
    development_consensus = pd.read_csv(
        OUTPUT_DIR / "xgboost_development_gene_consensus.csv"
    )
    signature_recovery = pd.read_csv(OUTPUT_DIR / "locked_signature_recovery.csv")
    specificity = pd.read_csv(
        OUTPUT_DIR / "locked_signature_specificity_summary.csv"
    )
    overlap = pd.read_csv(OUTPUT_DIR / "panel_overlap.csv")
    direct_signature_contrast = pd.read_csv(
        OUTPUT_DIR / "InterGATE_minus_XGBoost_signature_contrast.csv"
    )
    primary_programmes = pd.read_csv(
        OUTPUT_DIR / "prespecified_basal_programmes.csv"
    )
    cohort_weighting_sensitivity = pd.read_csv(
        OUTPUT_DIR / "xgboost_cohort_weighting_sensitivity.csv"
    )
else:
    runtime_development = prepare_runtime(
        CFG,
        include_external=False,
        build_biology_evidence=False,
    )
    assert runtime_development["external_opened"] is False
    display(preflight_frozen_xgboost(runtime_development, BENCHMARK_ROOT))
    development = run_development_comparison(
        runtime_development,
        benchmark_root=BENCHMARK_ROOT,
        results_root=RESULTS_ROOT,
        biology_results_dir=BIOLOGY_RESULTS_DIR,
        resource_dir=RESOURCE_DIR,
        output_dir=OUTPUT_DIR,
        intergate_core_file=INTERGATE_CORE_FILE,
        n_matched_draws=N_MATCHED_DRAWS,
        random_seed=RANDOM_SEED,
        batch_size=TREE_SHAP_BATCH_SIZE,
        tree_shap_device=TREE_SHAP_DEVICE,
    )
    development_consensus = development["development_consensus"]
    signature_recovery = development["signature_recovery"]
    specificity = development["specificity"]
    overlap = development["overlap"]
    direct_signature_contrast = development["direct_signature_contrast"]
    primary_programmes = development["primary_programmes"]
    cohort_weighting_sensitivity = development["cohort_weighting_sensitivity"]
    del runtime_development, development
    gc.collect()

print("Genes XGBoost top-288:", int(development_consensus["size_matched_panel"].sum()))
print("Genes XGBoost estrictos 5/5:", int(development_consensus["strict_5of5_panel"].sum()))
display(cohort_weighting_sensitivity)
display(specificity)
display(signature_recovery.loc[
    signature_recovery["annotation_class"].eq("Basal-like"),
    [
        "panel", "panel_size", "overlap_n",
        "fold_over_hypergeom_expected", "hypergeom_fdr_global",
        "fold_over_matched_null", "matched_fdr_global",
    ],
])
display(primary_programmes.loc[
    :, [
        "panel", "collection", "term", "overlap_n",
        "fold_enrichment", "fdr", "significant_fdr_005",
    ],
].sort_values(["panel", "fdr", "term"]))
display(overlap)
display(direct_signature_contrast)



## 4. Validación externa del ranking ya bloqueado

No se reajusta XGBoost ni se modifica el panel. Se calcula la concordancia de
rango y de signo en el externo agrupado y en cada cohorte por separado.



In [ ]:
external_completion = OUTPUT_DIR / "EXTERNAL_VALIDATION_COMPLETE.json"
if RUN_EXTERNAL:
    if external_completion.is_file():
        print("La validación externa ya está completa; se reutiliza.")
        transportability = pd.read_csv(
            OUTPUT_DIR / "xgboost_external_transportability.csv"
        )
    else:
        runtime_external = prepare_runtime(
            CFG,
            include_external=True,
            build_biology_evidence=False,
        )
        assert runtime_external["external_opened"] is True
        external = run_external_validation(
            runtime_external,
            benchmark_root=BENCHMARK_ROOT,
            output_dir=OUTPUT_DIR,
            batch_size=TREE_SHAP_BATCH_SIZE,
            tree_shap_device=TREE_SHAP_DEVICE,
        )
        transportability = external["transportability"]
        del runtime_external, external
        gc.collect()
    display(transportability)
else:
    print("RUN_EXTERNAL=False: el panel de desarrollo queda bloqueado, pero no se cierra el análisis.")



## 5. Cierre, figura y paquete de revisión



In [ ]:
if RUN_EXTERNAL:
    outputs = finalize_comparison(OUTPUT_DIR)
    print("Comparación completa.")
    display(pd.Series(outputs, name="path").to_frame())
    summary = pd.read_csv(outputs["summary"])
    display(summary)
    print("\nComprima/suba este archivo para interpretar el resultado:")
    print(outputs["review_zip"].resolve())



## Referencias metodológicas

- Lundberg SM, et al. From local explanations to global understanding with
  explainable AI for trees. *Nature Machine Intelligence*. 2020;2:56–67.
  https://doi.org/10.1038/s42256-019-0138-9
- Subramanian A, et al. Gene set enrichment analysis: a knowledge-based
  approach for interpreting genome-wide expression profiles. *PNAS*.
  2005;102:15545–15550. https://doi.org/10.1073/pnas.0506580102
- Liberzon A, et al. The Molecular Signatures Database Hallmark Gene Set
  Collection. *Cell Systems*. 2015;1:417–425.
  https://doi.org/10.1016/j.cels.2015.12.004
